# 03 — Treinamento, avaliação e demonstração

**Ative a GPU:** Ambiente de execução → Alterar tipo → T4 GPU.
Os checkpoints vão para o Drive (`runs/`), então se o Colab desconectar dá para retomar.

In [ ]:
# === Configuração: rode sempre que abrir o notebook ===
REPO_URL = "https://github.com/Lucas-Gomes-Alcantara/Rota-Livre-Visao.git"  # <- troque pelo repositório da equipe
import os
from google.colab import drive, userdata
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/rota_livre"          # pasta persistente (o disco do Colab é apagado ao desconectar)
os.makedirs(DRIVE, exist_ok=True)
if not os.path.exists("/content/Rota-Livre-Visao"):
    !git clone -q $REPO_URL /content/Rota-Livre-Visao
%cd /content/Rota-Livre-Visao
!pip install -q -r requirements.txt

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
if not os.path.exists('configs/data.yaml'):
    !unzip -qo $DRIVE/processed.zip -d .
os.makedirs(f'{DRIVE}/runs', exist_ok=True)
if not os.path.islink('runs'):
    !rm -rf runs && ln -s $DRIVE/runs runs
!cat configs/train.yaml

## 1. Treino (YOLO26s, pesos COCO)
Se aparecer *CUDA out of memory*: `!python src/train.py --batch 8`. Se o Colab cair: rode a célula de retomada.

In [ ]:
!python src/train.py

In [ ]:
# Retomar treino interrompido
# !python src/train.py --resume

In [ ]:
!cat results/metrics/train_env.json
from IPython.display import Image as I, display
display(I('results/figures/train_results.png', width=1000))

## 2. Avaliação no TESTE (nunca visto) e no teste brasileiro

In [ ]:
!python src/evaluate.py --bench-cpu
if os.path.exists('configs/data_test_br.yaml'):
    !python src/evaluate.py --data configs/data_test_br.yaml --tag test_br

In [ ]:
import glob
for f in sorted(glob.glob('results/figures/eval_test_confusion_matrix*.png')):
    display(I(f, width=600))

## 3. Demonstração com imagens fora do treino
Usa imagens do teste (preferência: brasileiras).

In [ ]:
import random, pathlib
src = [l for l in open('data/processed/test_br.txt').read().split() if l] if os.path.exists('data/processed/test_br.txt') else []
src = src or [str(p) for p in pathlib.Path('data/processed/test/images').iterdir()]
random.seed(1); sample = random.sample(src, min(8, len(src)))
os.makedirs('demo_inputs', exist_ok=True)
for p in sample: os.system(f'cp "{p}" demo_inputs/')
!python src/predict.py --source demo_inputs --conf 0.25
for f in sorted(glob.glob('results/predictions/*_pred.jpg'))[:8]:
    display(I(f, width=700))

## 4. Interface ao vivo para o vídeo
Gera um link público temporário. Grave a tela enviando uma foto e mostrando o resultado.

In [ ]:
!python src/demo_app.py --share

## 5. Salvar resultados no Drive e registrar versões

In [ ]:
!pip freeze > results/metrics/pip_freeze.txt
!python -c "import ultralytics, torch; print('ultralytics', ultralytics.__version__, '| torch', torch.__version__)"
!mkdir -p $DRIVE/entrega && cp -r models results $DRIVE/entrega/
print('salvo em', DRIVE + '/entrega')